#  Detección de placas de automóviles en imágenes

Este trabajo aborda la detección automática de placas vehiculares en imágenes mediante tres variantes de la familia YOLO (YOLOv11-n, YOLOv11-s y YOLOv10-s), seleccionadas por su bajo costo computacional y reducido número de parámetros.

En este notebook se muestra el código utilizado para la realización del proyecto.


In [ ]:
# Importamos las librerías a utilizar.
import pandas as pd
import yaml
from pathlib import Path
from itertools import product
from sklearn.model_selection import KFold, train_test_split
from ultralytics import YOLO
import os
import glob, re, pandas as pd
import matplotlib.pyplot as plt
import matplotlib as mpl
import scienceplots
import random
import os
import shutil
import uuid
import time

# Selección de hiperparámetros para el entrenamiento.

Para la optimización (o selección) de estos hiperparámetros se empleó validación cruzada (cross-validation) con 5 folds. Los folds se generaron a partir de una partición aleatoria de aproximadamente el 10\% del conjunto de entrenamiento (2,117 imágenes), de modo que en tres folds había 1,693 imágenes para entrenamiento y 424 para validación, y en los dos restantes 1\,694 para entrenamiento y 423 para validación; este mismo particionado se utilizó para los tres modelos.

Para cada combinación se entrenó por tres épocas.


## Creación de los folds

In [ ]:
ROOT = Path(r"/Users/pepefv97/Downloads/DATOS_PLACAS/train")  # raíz original (!Modificar en caso de correr el codigo)
IMG_DIR = ROOT / "images"
LBL_DIR = ROOT / "labels"


imgs = sorted(IMG_DIR.rglob("*.jpg"))
df   = pd.DataFrame({"img": imgs})
train_df = df.sample(frac=0.10, random_state=123).reset_index(drop=True)


K = 5
kf = KFold(n_splits=K, shuffle=True, random_state=123)

In [ ]:
OUT_ROOT = Path("folds")                     # carpeta raíz donde viven todos los folds
OUT_ROOT.mkdir(exist_ok=True)

fold_paths = []

for fold_i, (tr, va) in enumerate(kf.split(train_df)):
    tr_imgs = train_df.iloc[tr]["img"].tolist()
    va_imgs = train_df.iloc[va]["img"].tolist()


    for split in ["train", "val"]:
        for sub in ["images", "labels"]:
            (fold_dir / split / sub).mkdir(parents=True, exist_ok=True)


    def link(img_path, dst_img_dir):
        src_img = Path(img_path)
        src_lbl = LBL_DIR / src_img.with_suffix(".txt").name
        dst_img = dst_img_dir / src_img.name
        dst_lbl = dst_img_dir.parent / "labels" / src_lbl.name
        os.link(src_img, dst_img)
        os.link(src_lbl, dst_lbl)

    for p in tr_imgs:
        link(p, fold_dir / "train" / "images")
    for p in va_imgs:
        link(p, fold_dir / "val" / "images")

    yaml.dump(
        {
            "train": str((fold_dir / "train" / "images").resolve()),
            "val":   str((fold_dir / "val"   / "images").resolve()),
            "nc": 1,
            "names": ["PLACA_AUTOMOVIL"]
        },
        open(fold_dir / "data.yaml", "w")
    )

    fold_paths.append(fold_dir / "data.yaml")   # guardar ruta para entrenos posteriores

### Cross Validation YOLO11N

In [ ]:
param_grid = {
    "optimizer": ["SGD", "AdamW"],
    "lr0": [0.001, 0.005],
    "weight_decay": [1e-4, 5e-4],
    "mosaic": [0.5, 1.0]
}
grid = [dict(zip(param_grid, v)) for v in product(*param_grid.values())]

params_train = dict(epochs=3, batch=32, imgsz=640,
              device="mps", freeze=10, plots=True, verbose=True)

fold_paths = ['folds/fold0/data.yaml','folds/fold1/data.yaml','folds/fold2/data.yaml','folds/fold3/data.yaml','folds/fold4/data.yaml']

In [ ]:
for cfg in grid:
    for fold_yaml in fold_paths:
        name = (
            f"{Path(fold_yaml).parent.name}"
            f"_opt-{cfg['optimizer']}"
            f"_lr{cfg['lr0']}"
            f"_wd{cfg['weight_decay']}"
            f"_mos{cfg['mosaic']}"
        )
        YOLO("yolo11n.pt").train(
            data=str(fold_yaml),
            project="cross_validation_hpt",
            name=name,
            **cfg,
            **params_train
        )

In [ ]:

import glob, re, pandas as pd
from pathlib import Path

ROOT_RUNS = Path("cross_validation_hpt")
pattern   = re.compile(
    r"fold\d+_"
    r"opt-(?P<opt>\w+)_"
    r"lr(?P<lr>[\d.]+)_"
    r"wd(?P<wd>[\d.]+)_"
    r"mos(?P<mos>[\d.]+)"
)

records = []

for csv in ROOT_RUNS.rglob("results.csv"):
    run_dir = csv.parent
    m = pattern.search(run_dir.name)
    if not m:
        continue
    hp = m.groupdict()
    df = pd.read_csv(csv)


    best_row        = df.loc[df["metrics/mAP50(B)"].idxmax()]
    best_map50      = best_row["metrics/mAP50(B)"]
    best_map5095    = best_row["metrics/mAP50-95(B)"]

    records.append({
        **hp,
        "fold"     : run_dir.name.split("_")[0],
        "mAP50"    : best_map50,
        "mAP50_95" : best_map5095
    })


df = pd.DataFrame(records)
group_cols = ["opt", "lr", "wd", "mos"]

summary = (
    df.groupby(group_cols)
      .agg(
          mean    = ("mAP50",    "mean"),
          std     = ("mAP50",    "std"),
          count   = ("fold",     "count"),
          mean95  = ("mAP50_95", "mean"),
          std95   = ("mAP50_95", "std")
      )
      .reset_index()
      .sort_values("mean", ascending=False)
)

print(summary.head(5))


### YOLO10S

In [ ]:
param_grid = {
    "optimizer":     ["SGD", "AdamW"],
    "lr0":           [1e-3, 5e-4],
    "weight_decay":  [1e-4, 5e-4],
    "mosaic":        [0,0.5]
}

grid = [dict(zip(param_grid, v)) for v in product(*param_grid.values())]

In [ ]:
params_train = dict(
    epochs=3,
    batch=32,
    imgsz=512,
    device="mps",
    freeze=10,
    plots=True,
    verbose=True,
    amp=True
)

fold_paths = ['folds/fold0/data.yaml','folds/fold1/data.yaml','folds/fold2/data.yaml','folds/fold3/data.yaml','folds/fold4/data.yaml']

In [ ]:

for cfg in grid:
    for fold_yaml in fold_paths:
        name = (
            f"{Path(fold_yaml).parent.name}"
            f"_opt-{cfg['optimizer']}"
            f"_lr{cfg['lr0']}"
            f"_wd{cfg['weight_decay']}"
            f"_mos{cfg['mosaic']}"
        )
        YOLO("yolov10s.pt").train(
            data=str(fold_yaml),
            project="yolov10_cross_validation_hpt",
            name=name,
            **cfg,
            **params_train
        )

In [ ]:

ROOT_RUNS = Path("yolov10_cross_validation_hpt")
pattern   = re.compile(
    r"fold\d+_"
    r"opt-(?P<opt>\w+)_"
    r"lr(?P<lr>[\d.]+)_"
    r"wd(?P<wd>[\d.]+)_"
    r"mos(?P<mos>[\d.]+)"
)

records = []

for csv in ROOT_RUNS.rglob("results.csv"):
    run_dir = csv.parent
    m = pattern.search(run_dir.name)
    if not m:
        continue
    hp = m.groupdict()
    df = pd.read_csv(csv)

    best_row        = df.loc[df["metrics/mAP50(B)"].idxmax()]
    best_map50      = best_row["metrics/mAP50(B)"]
    best_map5095    = best_row["metrics/mAP50-95(B)"]

    records.append({
        **hp,
        "fold"     : run_dir.name.split("_")[0],
        "mAP50"    : best_map50,
        "mAP50_95" : best_map5095
    })


df = pd.DataFrame(records)
group_cols = ["opt", "lr", "wd", "mos"]

summary = (
    df.groupby(group_cols)
      .agg(
          mean    = ("mAP50",    "mean"),
          std     = ("mAP50",    "std"),
          count   = ("fold",     "count"),
          mean95  = ("mAP50_95", "mean"),
          std95   = ("mAP50_95", "std")
      )
      .reset_index()
      .sort_values("mean", ascending=False)
)

print(summary.head(5))


### YOLO11S

In [ ]:
param_grid = {
    "optimizer":     ["SGD", "AdamW"],
    "lr0":           [1e-3, 5e-4],
    "weight_decay":  [1e-4, 5e-4],
    "mosaic":        [0,0.5]
}

grid = [dict(zip(param_grid, v)) for v in product(*param_grid.values())]

In [ ]:
params_train = dict(
    epochs=3,
    batch=32,
    imgsz=512,
    device="mps",
    freeze=10,
    plots=True,
    verbose=True,
    amp=True
)

fold_paths = ['folds/fold0/data.yaml','folds/fold1/data.yaml','folds/fold2/data.yaml','folds/fold3/data.yaml','folds/fold4/data.yaml']

In [ ]:

for cfg in grid:
    for fold_yaml in fold_paths:
        name = (
            f"{Path(fold_yaml).parent.name}"
            f"_opt-{cfg['optimizer']}"
            f"_lr{cfg['lr0']}"
            f"_wd{cfg['weight_decay']}"
            f"_mos{cfg['mosaic']}"
        )
        YOLO("yolo11s.pt").train(
            data=str(fold_yaml),
            project="yolov11_cross_validation_hpt",
            name=name,
            **cfg,
            **params_train
        )

In [ ]:
import glob, re, pandas as pd
from pathlib import Path

ROOT_RUNS = Path("yolov11_cross_validation_hpt")
pattern   = re.compile(
    r"fold\d+_"
    r"opt-(?P<opt>\w+)_"
    r"lr(?P<lr>[\d.]+)_"
    r"wd(?P<wd>[\d.]+)_"
    r"mos(?P<mos>[\d.]+)"
)

records = []

for csv in ROOT_RUNS.rglob("results.csv"):
    run_dir = csv.parent
    m = pattern.search(run_dir.name)
    if not m:
        continue
    hp = m.groupdict()
    df = pd.read_csv(csv)

    best_row        = df.loc[df["metrics/mAP50(B)"].idxmax()]
    best_map50      = best_row["metrics/mAP50(B)"]
    best_map5095    = best_row["metrics/mAP50-95(B)"]

    records.append({
        **hp,
        "fold"     : run_dir.name.split("_")[0],
        "mAP50"    : best_map50,
        "mAP50_95" : best_map5095
    })

df = pd.DataFrame(records)
group_cols = ["opt", "lr", "wd", "mos"]

summary = (
    df.groupby(group_cols)
      .agg(
          mean    = ("mAP50",    "mean"),
          std     = ("mAP50",    "std"),
          count   = ("fold",     "count"),
          mean95  = ("mAP50_95", "mean"),
          std95   = ("mAP50_95", "std")
      )
      .reset_index()
      .sort_values("mean", ascending=False)
)

print(summary.head(5))


# Entrenamiento

Con base en los resultados del *cross‐validation*, para cada variante se eligió la combinación de hiperparámetros que maximiza la métrica $mAP@0.5$.  
De este modo, los ajustes finales quedaron definidos de la siguiente manera:

- **YOLOv11-n**: `batch_size` = 32, resolución de 640 px, `freeze` = 10, optimizador *AdamW*, $lr_{0} = 1 \times 10^{-3}$, `weight_decay` = $1 \times 10^{-4}$ y `mosaic` = 1.0.  
- **YOLOv11-s** / **YOLOv10-s**: `batch_size` = 32, resolución de 512 px, `freeze` = 10, optimizador *AdamW*, $lr_{0} = 5 \times 10^{-4}$, `weight_decay` = $5 \times 10^{-4}$, `mosaic` = 0 y `amp` = True.

Los modelos a entrenar son las versiones preentrenadas en COCO que ofrece Ultralytics

## YOLO11N

In [ ]:
from ultralytics import YOLO

path_yml = r'/Users/pepefv97/Downloads/DATOS_PLACAS/data.yaml'


model = YOLO("yolo11n.pt")
model.train(
    data=path_yml,
    epochs=10,
    batch=32,
    imgsz=640,
    device="mps",
    optimizer="AdamW",
    lr0=0.001,
    weight_decay=0.0001,
    mosaic=1.0,
    freeze=10,
    plots=True,
    project="ENTRENAMIENTO_YOLOV11",
    name="YOLOV11_AdamW_lr0.001_wd1e-4_mos1.0",
    verbose=True
)

## YOLO10S

In [ ]:
from ultralytics import YOLO

path_yml = r'/Users/pepefv97/Downloads/DATOS_PLACAS/data.yaml'


model = YOLO("yolov10s.pt")
model.train(
    data=path_yml,
    epochs=10,
    batch=32,
    imgsz=512,
    device="mps",
    freeze=10,
    plots=True,
    verbose=True,
    amp=True,
    optimizer="AdamW",
    lr0=0.0005,
    weight_decay=0.0005,
    mosaic=0,
    project="ENTRENAMIENTO_YOLOV10S",
    name="YOLOV10S_AdamW_lr0.0005_wd0.0005_mos0"
)

## YOLO11S

In [ ]:
from ultralytics import YOLO

path_yml = r'/Users/pepefv97/Downloads/DATOS_PLACAS/data.yaml'


model = YOLO("yolo11s.pt")
model.train(
    data=path_yml,
    epochs=10,
    batch=32,
    imgsz=512,
    device="mps",
    freeze=10,
    plots=True,
    verbose=True,
    amp=True,
    optimizer="AdamW",
    lr0=0.0005,
    weight_decay=0.0005,
    mosaic=0,
    project="ENTRENAMIENTO_YOLOV11S",
    name="YOLOV11S_AdamW_lr0.0005_wd0.0005_mos0"
)

# Gráficas de evaluación de métricas durante el entrenamiento.

En las gráficas se ilustra, respectivamente, el progreso de mAP@0.5, mAP@0.5:0.95 y el tiempo de entrenamiento acumulado por época, proporcionando una visión clara de la convergencia y eficiencia de cada variante.

In [ ]:

mpl.rcdefaults()
plt.style.use(['science', 'no-latex', 'grid'])


mpl.rcParams.update({
    "text.usetex": False,
    "font.family": "sans-serif",
    "font.sans-serif": ["DejaVu Sans"],
    "mathtext.fontset": "dejavusans",
    "mathtext.default": "regular",
    "axes.unicode_minus": False,
    "figure.dpi": 300,
    "font.size": 10,
})




csv_paths = {
    "YOLOv11-n": Path("/Users/pepefv97/Downloads/PROYECTO_5/ENTRENAMIENTO_YOLOV11/YOLOV11_AdamW_lr0.001_wd1e-4_mos1.0/results.csv"),
    "YOLOv11-s": Path("/Users/pepefv97/Downloads/PROYECTO_5/ENTRENAMIENTO_YOLOV11S/YOLOV11S_AdamW_lr0.0005_wd0.0005_mos0/results.csv"),
    "YOLOv10-s": Path("/Users/pepefv97/Downloads/PROYECTO_5/ENTRENAMIENTO_YOLOV10S/YOLOV10S_AdamW_lr0.0005_wd0.0005_mos0/results.csv")
}


frames = []
for label, path in csv_paths.items():
    df = pd.read_csv(
        path
    )
    df["model"] = label
    frames.append(df)

data = pd.concat(frames, ignore_index=True)




pro_palette = ["#3C5488",
               "#E64B35",
               "#00A087"]

line_styles = ["-", "--", ":"]




In [ ]:
fig, ax = plt.subplots(figsize=(4.8, 3.4))
for i, (label, grp) in enumerate(data.groupby("model")):
        ax.plot(
            grp["epoch"],
            grp["metrics/mAP50(B)"],
            label=label,
            color=pro_palette[i % len(pro_palette)],
            linestyle=line_styles[i % len(line_styles)],
            marker='o',
            markersize=3
        )
ax.set_xlabel("Época")
ax.set_ylabel("mAP@0.5")
ax.set_title("Métrica mAP@0.5 vs Época", pad=6)
ax.legend(frameon=False)
fig.tight_layout()
plt.show()

fig.savefig("map50_vs_epoca.png", dpi=300, bbox_inches="tight")

In [ ]:
fig, ax = plt.subplots(figsize=(4.8, 3.4))
for i, (label, grp) in enumerate(data.groupby("model")):
        ax.plot(
            grp["epoch"],
            grp["metrics/mAP50-95(B)"],
            label=label,
            color=pro_palette[i % len(pro_palette)],
            linestyle=line_styles[i % len(line_styles)],
            marker='o',
            markersize=3
        )
ax.set_xlabel("Época")
ax.set_ylabel("mAP@0.5:0.95")
ax.set_title("Métrica mAP@0.5:0.95 vs Época", pad=6)
ax.legend(frameon=False)
fig.tight_layout()
plt.show()

fig.savefig("map5095_vs_epoca.png", dpi=300, bbox_inches="tight")



In [ ]:
fig, ax = plt.subplots(figsize=(4.8, 3.4))
for i, (label, grp) in enumerate(data.groupby("model")):
        ax.plot(
            grp["epoch"],
            grp["time"]/60,
            label=label,
            color=pro_palette[i % len(pro_palette)],
            linestyle=line_styles[i % len(line_styles)],
            marker='o',
            markersize=3
        )
ax.set_xlabel("Época")
ax.set_ylabel("Tiempo cumulado ([minutos])")
ax.set_title("Tiempo acumulado de entrenamiento vs Época", pad=6)
ax.legend(frameon=False)
fig.tight_layout()
plt.show()

fig.savefig("tiempo_vs_epoca.png", dpi=300, bbox_inches="tight")


# Evaluación de los modelos

A fin de cuantificar la variabilidad o incertidumbre estadística de los resultados, se aplicó un procedimiento de *bootstrap* con 1,000 réplicas del conjunto de prueba original (1,019 imágenes por réplica), generadas mediante muestreo aleatorio con reemplazo.  

Para cada réplica se calcularon $mAP@0.5$ y $mAP@0.5:0.95$; a partir de las 1,000 estimaciones se obtuvieron la mediana y los intervalos de confianza al 95\%.


## Generación de 1,000 replicas de conjnunto de prueba

In [ ]:



B  = 1000
TEST_IMG = pathlib.Path("/Users/pepefv97/Downloads/DATOS_PLACAS/test/images")
TEST_LAB = pathlib.Path("/Users/pepefv97/Downloads/DATOS_PLACAS/test/labels")
BOOT_ROOT = pathlib.Path("BOOTSTRAP/SETS")
CLASS_NAME = ["PLACA_AUTOMOVIL"]
ROOT_PATH = pathlib.Path("/Users/pepefv97/Downloads/PROYECTO_5")
imgs = sorted(TEST_IMG.glob("*.jpg"))


random.seed(123)

for b in range(B):
    set_dir  = ROOT_PATH / BOOT_ROOT / f"set_{b:04d}"
    img_dir  = ROOT_PATH / set_dir / "test/images"
    lab_dir  = ROOT_PATH / set_dir / "test/labels"
    img_dir.mkdir(parents=True, exist_ok=True)
    lab_dir.mkdir(parents=True, exist_ok=True)

    sample = random.choices(imgs, k=len(imgs))
    for idx, src in enumerate(sample):
        tag      = f"{idx:04d}_{uuid.uuid4().hex[:6]}"
        dst_img  = img_dir / f"{tag}_{src.name}"
        dst_lab  = lab_dir / f"{tag}_{src.stem}.txt"

        os.symlink(src.resolve(), dst_img)      # enlace a la imagen
        os.symlink((TEST_LAB / f"{src.stem}.txt").resolve(), dst_lab)

    yaml_path = set_dir / "data.yaml"
    yaml_path.write_text(yaml.dump({
        "names": CLASS_NAME,
        "nc"   : len(CLASS_NAME),
        "train" : str(img_dir),
        "val" : str(img_dir),
        "test" : str(img_dir)
    }))


## Evaluación de los tres modelos

In [ ]:
BOOT_SETS = ROOT_PATH / BOOT_ROOT
OUT_ROOT = ROOT_PATH / pathlib.Path("BOOTSTRAP")
DEVICE = "mps"
BATCH = 64


MODELS = {
    "YOLOV11N": "ENTRENAMIENTO_YOLOV11/YOLOV11_AdamW_lr0.001_wd1e-4_mos1.0/weights/best.pt",
    "YOLOV11S": "ENTRENAMIENTO_YOLOV11S/YOLOV11S_AdamW_lr0.0005_wd0.0005_mos0/weights/best.pt",
    "YOLOV11M": "ENTRENAMIENTO_YOLOV10S/YOLOV10S_AdamW_lr0.0005_wd0.0005_mos0/weights/best.pt"
}

yaml_files = sorted(BOOT_SETS.glob("set_*/data.yaml"))
assert yaml_files, "No se encontraron réplicas en BOOTSTRAP/SETS/"

rows = []
for model_name, model_path in MODELS.items():
    print(f"Evaluando {model_name} …")
    model = YOLO(model_path)

    for yml in yaml_files:
        start = time.time()
        r = model.val(
            data=str(yml),
            split="test",
            batch=BATCH,
            device=DEVICE,
            workers=0,
            verbose=False,
            save=False,
            plots=False
        )

        rows.append({
            "Model"   : model_name,
            "Sample"  : yml.parent.name,
            "Precision": r.box.mp,
            "Recall"   : r.box.mr,
            "mAP50"    : r.box.map50,
            "mAP5095"  : r.box.map,
            "Pre_ms"   : r.speed["preprocess"],
            "Inf_ms"   : r.speed["inference"],
            "Post_ms"  : r.speed["postprocess"],
            "Elapsed_s": round(time.time() - start, 2)
        })

    out_dir = OUT_ROOT / model_name
    out_dir.mkdir(exist_ok=True)
    (pd.DataFrame([r for r in rows if r["Model"] == model_name])
        .drop(columns="Model")
        .to_csv(out_dir / "bootstrap_metrics.csv", index=False))

print("CSV listos en BOOTSTRAP/<MODEL>/bootstrap_metrics.csv")


## Cálculo de mediana e intervalos de confianza al 95% para $mAP@0.5$ y $mAP@0.5:0.95$.

In [ ]:
csv_files = {
    "YOLOv11-n": Path("/Users/pepefv97/Downloads/PROYECTO_5/BOOTSTRAP/YOLOV11N/bootstrap_metrics.csv"),
    "YOLOv11-s": Path("/Users/pepefv97/Downloads/PROYECTO_5/BOOTSTRAP/YOLOV11S/bootstrap_metrics.csv"),
    "YOLOv10-s": Path("/Users/pepefv97/Downloads/PROYECTO_5/BOOTSTRAP/YOLOV10S/bootstrap_metrics.csv"),
}


metric_cols = [
    "Precision", "Recall",
    "mAP50", "mAP5095",
    "Pre_ms", "Inf_ms", "Post_ms"
]

rows = []
for model, path in csv_files.items():
    df = pd.read_csv(path, usecols=metric_cols)
    ci_low  = df.quantile(0.025)
    ci_high = df.quantile(0.975)
    median  = df.median()

    for col in metric_cols:
        rows.append({
            "Model": model,
            "Metric": col,
            "Median": median[col],
            "CI_low": ci_low[col],
            "CI_high": ci_high[col]
        })

summary = pd.DataFrame(rows)

In [ ]:

csv_files = {
    "YOLOv11-n": Path("/Users/pepefv97/Downloads/PROYECTO_5/BOOTSTRAP/YOLOV11N/bootstrap_metrics.csv"),
    "YOLOv11-s": Path("/Users/pepefv97/Downloads/PROYECTO_5/BOOTSTRAP/YOLOV11S/bootstrap_metrics.csv"),
    "YOLOv10-s": Path("/Users/pepefv97/Downloads/PROYECTO_5/BOOTSTRAP/YOLOV10S/bootstrap_metrics.csv"),
}

rows = []
for model, path in csv_files.items():
    df = pd.read_csv(path)


    df["Time_ms"] = df["Pre_ms"] + df["Inf_ms"] + df["Post_ms"]
    inf_q = df["Inf_ms"].quantile([0.025, 0.50, 0.975])
    tot_q = df["Time_ms"].quantile([0.025, 0.50, 0.975])

    rows.append({
        "Model": model,
        "Inf_median":  inf_q.loc[0.50],
        "Inf_CI_low":  inf_q.loc[0.025],
        "Inf_CI_high": inf_q.loc[0.975],
        "Total_median":  tot_q.loc[0.50],
        "Total_CI_low":  tot_q.loc[0.025],
        "Total_CI_high": tot_q.loc[0.975],
    })

summary = pd.DataFrame(rows)
print(summary)
